In [7]:
!pip install dtreeviz graphviz ipykernel cairosvg

In [8]:
import dtreeviz
import graphviz

In [9]:
import os

graphviz_bin = r'C:\Program Files\Graphviz\bin'

os.environ['PATH'] = graphviz_bin + os.pathsep + os.environ['PATH']

In [10]:
import matplotlib as mpl
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

mpl.rcParams['svg.fonttype'] = 'none'

In [11]:
%matplotlib inline

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import warnings
import time

from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

In [12]:
plt.style.use("ggplot")

In [13]:
df = pd.read_csv('jeju_bus.csv')

display(df.head())

print('shape: ' , df.shape)
print('전체 결측치 개수: ', df.isnull().sum().sum())

,id,date,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,now_arrive_time,distance,next_station,next_latitude,next_longitude,next_arrive_time
0,0,2019-10-15,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,06시,266.0,제대마을,33.457724,126.554014,24
1,1,2019-10-15,405136001,7997025,360-1,33.457724,126.554014,제대마을,06시,333.0,제대아파트,33.458783,126.557353,36
2,2,2019-10-15,405136001,7997025,360-1,33.458783,126.557353,제대아파트,06시,415.0,제주대학교,33.459893,126.561624,40
3,3,2019-10-15,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),06시,578.0,제주여자중고등학교(아라방면),33.484860,126.542928,42
4,4,2019-10-15,405136001,7997025,360-1,33.485662,126.494923,도호동,07시,374.0,은남동,33.485822,126.490897,64


shape:  (210457, 14)
전체 결측치 개수:  0


In [14]:
df_model = df.copy()

df_model['original_index'] = df_model.index

target_col = 'next_arrive_time'

In [15]:
df_model['date'] = pd.to_datetime(df_model['date'])

df_model['day'] = df_model['date'].dt.day
df_model['dayofweek'] = df_model['date'].dt.dayofweek

df_model['now_hour'] = (
        df_model['now_arrive_time'].astype(str).str.extract(r"(\d+)").astype(float) # extract 목적: 한 그룹만 뽑아서 변환이기 때문에 괄호로 묶음
)

df_model[['date', 'day', 'dayofweek', 'now_arrive_time', 'now_hour']].head()

,date,day,dayofweek,now_arrive_time,now_hour
0,2019-10-15,15,1,06시,6.0
1,2019-10-15,15,1,06시,6.0
2,2019-10-15,15,1,06시,6.0
3,2019-10-15,15,1,06시,6.0
4,2019-10-15,15,1,07시,7.0


In [16]:
df_model['station_segment'] = (
    df_model['now_station'].astype(str) + ' -> ' + df_model['next_station'].astype(str)
)

df_model.head()

,id,date,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,now_arrive_time,distance,next_station,next_latitude,next_longitude,next_arrive_time,original_index,day,dayofweek,now_hour,station_segment
0,0,2019-10-15,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,06시,266.0,제대마을,33.457724,126.554014,24,0,15,1,6.0,제주대학교입구 -> 제대마을
1,1,2019-10-15,405136001,7997025,360-1,33.457724,126.554014,제대마을,06시,333.0,제대아파트,33.458783,126.557353,36,1,15,1,6.0,제대마을 -> 제대아파트
2,2,2019-10-15,405136001,7997025,360-1,33.458783,126.557353,제대아파트,06시,415.0,제주대학교,33.459893,126.561624,40,2,15,1,6.0,제대아파트 -> 제주대학교
3,3,2019-10-15,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),06시,578.0,제주여자중고등학교(아라방면),33.484860,126.542928,42,3,15,1,6.0,남국원(아라방면) -> 제주여자중고등학교(아라방면)
4,4,2019-10-15,405136001,7997025,360-1,33.485662,126.494923,도호동,07시,374.0,은남동,33.485822,126.490897,64,4,15,1,7.0,도호동 -> 은남동


In [17]:
print('station_segment 고유값 개수: ', df_model['station_segment'].nunique())

station_segment 고유값 개수:  724


In [18]:
def calculate_distance_km(lat1, lon1, lat2, lon2):
    
    earth_radius_km = 6371

    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)

    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)

    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    a = (np.sin(dlat/2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon/2)**2)
    c = 2 * np.arcsin(np.sqrt(a))
    
    return earth_radius_km * c

In [19]:
reference_points = {
    "up": (33.506286, 126.490312),      # 제주공항
    "down": (33.246742, 126.562387),    # 서귀포시 근처
    "right": (33.493521, 126.895326),    # 성산일출봉 방면
    "center": (33.379726, 126.545315)   # 한라산 / 중산간
}

In [20]:
def assign_region_info(data, lat_col, lon_col):
    up_lat, up_lon = reference_points['up']
    down_lat, down_lon = reference_points['down']
    right_lat, right_lon = reference_points['right']
    center_lat, center_lon = reference_points['center']

    distance_to_up = calculate_distance_km(data[lat_col], data[lon_col], up_lat, up_lon)
    distance_to_down = calculate_distance_km(data[lat_col], data[lon_col], down_lat, down_lon)
    distance_to_right = calculate_distance_km(data[lat_col], data[lon_col], right_lat, right_lon)
    distance_to_center = calculate_distance_km(data[lat_col], data[lon_col], center_lat, center_lon)

    distance_table = pd.DataFrame({
        'up': distance_to_up,
        'down': distance_to_down,
        'right': distance_to_right,
        'center': distance_to_center
    }, index=data.index)

    nearest_region = distance_table.idxmin(axis=1)

    result = pd.DataFrame({
        'dist_name': nearest_region,
        'dist_to_up': distance_to_up,
        'dist_to_down': distance_to_down,
        'dist_to_right': distance_to_right,
        'dist_to_center': distance_to_center
    }, index=data.index)

    return result

In [21]:
now_region_info = assign_region_info(
    df_model,
    'now_latitude',
    'now_longitude'
)

now_region_info.head()

,dist_name,dist_to_up,dist_to_down,dist_to_right,dist_to_center
0,up,7.962505,23.319056,32.135034,8.531900
1,up,8.003869,23.473015,31.905930,8.710479
2,up,8.158338,23.582519,31.583874,8.861451
3,up,5.774762,25.961685,32.635035,11.118034
4,up,2.332803,27.295447,37.141639,12.673763


In [22]:
next_region_info = assign_region_info(
    df_model,
    'next_latitude',
    'next_longitude'
)

next_region_info.head()

,dist_name,dist_to_up,dist_to_down,dist_to_right,dist_to_center
0,up,8.003869,23.473015,31.905930,8.710479
1,up,8.158338,23.582519,31.583874,8.861451
2,up,8.387595,23.701416,31.175511,9.041759
3,up,5.429627,26.539110,32.693959,11.692466
4,up,2.276139,27.400941,37.514443,12.832663


In [23]:
df_model['now_dist_name'] = now_region_info['dist_name']
df_model['now_dist_to_up'] = now_region_info['dist_to_up']
df_model['now_dist_to_down'] = now_region_info['dist_to_down']
df_model['now_dist_to_right'] = now_region_info['dist_to_right']
df_model['now_dist_to_center'] = now_region_info['dist_to_center']

df_model.head()

,id,date,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,now_arrive_time,distance,...,original_index,day,dayofweek,now_hour,station_segment,now_dist_name,now_dist_to_up,now_dist_to_down,now_dist_to_right,now_dist_to_center
0,0,2019-10-15,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,06시,266.0,...,0,15,1,6.0,제주대학교입구 -> 제대마을,up,7.962505,23.319056,32.135034,8.531900
1,1,2019-10-15,405136001,7997025,360-1,33.457724,126.554014,제대마을,06시,333.0,...,1,15,1,6.0,제대마을 -> 제대아파트,up,8.003869,23.473015,31.905930,8.710479
2,2,2019-10-15,405136001,7997025,360-1,33.458783,126.557353,제대아파트,06시,415.0,...,2,15,1,6.0,제대아파트 -> 제주대학교,up,8.158338,23.582519,31.583874,8.861451
3,3,2019-10-15,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),06시,578.0,...,3,15,1,6.0,남국원(아라방면) -> 제주여자중고등학교(아라방면),up,5.774762,25.961685,32.635035,11.118034
4,4,2019-10-15,405136001,7997025,360-1,33.485662,126.494923,도호동,07시,374.0,...,4,15,1,7.0,도호동 -> 은남동,up,2.332803,27.295447,37.141639,12.673763


In [24]:
df_model['next_dist_name'] = next_region_info['dist_name']
df_model['next_dist_to_up'] = next_region_info['dist_to_up']
df_model['next_dist_to_down'] = next_region_info['dist_to_down']
df_model['next_dist_to_right'] = next_region_info['dist_to_right']
df_model['next_dist_to_center'] = next_region_info['dist_to_center']

df_model.head()

,id,date,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,now_arrive_time,distance,...,now_dist_name,now_dist_to_up,now_dist_to_down,now_dist_to_right,now_dist_to_center,next_dist_name,next_dist_to_up,next_dist_to_down,next_dist_to_right,next_dist_to_center
0,0,2019-10-15,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,06시,266.0,...,up,7.962505,23.319056,32.135034,8.531900,up,8.003869,23.473015,31.905930,8.710479
1,1,2019-10-15,405136001,7997025,360-1,33.457724,126.554014,제대마을,06시,333.0,...,up,8.003869,23.473015,31.905930,8.710479,up,8.158338,23.582519,31.583874,8.861451
2,2,2019-10-15,405136001,7997025,360-1,33.458783,126.557353,제대아파트,06시,415.0,...,up,8.158338,23.582519,31.583874,8.861451,up,8.387595,23.701416,31.175511,9.041759
3,3,2019-10-15,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),06시,578.0,...,up,5.774762,25.961685,32.635035,11.118034,up,5.429627,26.539110,32.693959,11.692466
4,4,2019-10-15,405136001,7997025,360-1,33.485662,126.494923,도호동,07시,374.0,...,up,2.332803,27.295447,37.141639,12.673763,up,2.276139,27.400941,37.514443,12.832663


In [25]:
# TODO: p33부터 시작
df_model['dist_segment_name'] = (
    df_model['now_dist_name'].astype(str)
    + ' -> '
    + df_model['next_dist_name'].astype(str)
)

print('dist_segment_name 고유값 개수: ', df_model['dist_segment_name'].nunique())
df_model['dist_segment_name'].value_counts()

dist_segment_name 고유값 개수:  12


dist_segment_name
up -> up            121131
down -> down         48377
right -> right       34032
center -> center      5512
center -> up           321
right -> up            276
up -> center           207
down -> right          144
up -> right            137
right -> down          134
down -> center          95
center -> down          91
Name: count, dtype: int64

In [26]:
df_model.columns

Index(['id', 'date', 'route_id', 'vh_id', 'route_nm', 'now_latitude',
       'now_longitude', 'now_station', 'now_arrive_time', 'distance',
       'next_station', 'next_latitude', 'next_longitude', 'next_arrive_time',
       'original_index', 'day', 'dayofweek', 'now_hour', 'station_segment',
       'now_dist_name', 'now_dist_to_up', 'now_dist_to_down',
       'now_dist_to_right', 'now_dist_to_center', 'next_dist_name',
       'next_dist_to_up', 'next_dist_to_down', 'next_dist_to_right',
       'next_dist_to_center', 'dist_segment_name'],
      dtype='str')

In [47]:
selected_numeric_features = [
        'distance', 'day', 'dayofweek', 'now_hour',
        'now_dist_to_up', 'now_dist_to_down',
       'now_dist_to_right', 'now_dist_to_center',
       'next_dist_to_up', 'next_dist_to_down', 'next_dist_to_right',
       'next_dist_to_center'
]

In [48]:
selected_categorical_features = [
    'route_nm', 'now_station', 'next_station', 'dist_segment_name'
]

In [49]:
selected_features = selected_numeric_features + selected_categorical_features

selected_features

['distance',
 'day',
 'dayofweek',
 'now_hour',
 'now_dist_to_up',
 'now_dist_to_down',
 'now_dist_to_right',
 'now_dist_to_center',
 'next_dist_to_up',
 'next_dist_to_down',
 'next_dist_to_right',
 'next_dist_to_center',
 'route_nm',
 'now_station',
 'next_station',
 'dist_segment_name']

In [50]:
upper_1pct = df_model[target_col].quantile(0.99)

upper_1pct

np.float64(340.0)

In [51]:
df_model_selected = df_model[df_model[target_col] <= upper_1pct].copy()

In [55]:
X = df_model_selected[selected_features]

In [57]:
y = df_model_selected[target_col]

In [58]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,
    random_state=42
)

In [ ]:
preprocessor = ColumnTransformer(
                    transformers=[
                        ('cat',
                        OneHotEncoder(handle_unknown='ignore'),
                        selected_categorical_features)
                    ],
                    remainder='passthrough'
                )

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{fea

In [65]:
def evaluate_regression_model(model_name, description, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    print(f'[{model_name}] MAE: {mae:.4f} | RMSE: {rmse:.4f}')

    return {
        'model_name': model_name,
        'description': description,
        'MAE': mae, 'RMSE': rmse
    }

In [68]:
def my_sum(num1, num2, num3):
    print(f'num1={num2}, num2={num2}, num3={num3}')
    return num1+num2+num3

In [72]:
param = {'num3': 100, 'num2': 200, 'num1': 500}

my_sum(**param)

num1=200, num2=200, num3=100


800

In [69]:
my_sum(10,20,30)

num1=20, num2=20, num3=30


60

In [73]:
def train_xgb_pipeline(model_name1, description1, xgb_params):
    xgb_model = XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1,
        **xgb_params
    )

    model_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', xgb_model)
    ])
    model_pipeline.fit(X_train, y_train)

    y_pred1 = model_pipeline.predict(X_test)

    evaluate_regression_model(
        model_name=model_name1,
        description=description1,
        y_true=y_test,
        y_pred=y_pred1
    )

    return {
        'model_name': model_name1,
        'pipeline': model_pipeline,
        'y_pred': y_pred1,
        'evaluation': evaluation
    }

In [82]:
param_di = {
    'model__max_depth': [3, 5, 7, 9],
    'model__learning_rate': [0.03, 0.05, 0.1],
    'model__n_estimators': [200, 300, 400]
}

scoring = 'neg_mean_absolute_error'

In [83]:
search_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        n_jobs=1
    ))
])

In [89]:
from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=search_model,
    param_distributions=param_di,
    n_iter=25,
    scoring='neg_mean_absolute_error',
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [94]:
random_search.fit(X_train, y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__learning_rate': [0.03, 0.05, ...], 'model__max_depth': [3, 5, ...], 'model__n_estimators': [200, 300, ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",25
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used he

In [92]:
random_search.best_params_

{'model__n_estimators': 400,
 'model__max_depth': 9,
 'model__learning_rate': 0.1}

In [93]:
random_search.best_score_

np.float64(-18.88917465209961)